# Hostamar — Free GPU Backend (Google Colab variant)

This notebook is **functionally identical** to `kaggle-animatediff-server.ipynb`
but uses the Colab runtime. Pick whichever gives you more GPU time.

## Quick start

1. Open in Google Colab: `File → Open Notebook → GitHub → monjilaktn/hostamar/notebooks/`
2. **Runtime → Change runtime type → GPU → T4**
3. Run cells 1-6 in order. After Cell 4 you'll see a public URL like:
   `https://xxxx-xx-xxx-xx-xxx.ngrok-free.app`
4. Copy that URL into your `hostamar.com` `.env.local` as:
   ```
   COLAB_VIDEO_URL=https://xxxx-xx-xxx-xx-xxx.ngrok-free.app
   ```
5. Uncomment "colab" in `workers/video_worker_gpu.py` `PROVIDER_PRIORITY` and restart.

## Quota

- Colab free: ~12 hours/day, **disconnects often** (good for testing, not production).
- Colab Pro ($10/mo): more reliable but still no SLA.
- For 24/7 operation use Kaggle (30h/week, persistent).


In [ ]:
# Cell 1: confirm GPU
import torch
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free // 1024**2} MB free / {total // 1024**2} MB total")


In [ ]:
# Cell 2: install deps (Colab has torch pre-installed for GPU; just add HF + FastAPI)
!pip install -q diffusers==0.30.0 transformers==4.44.2 accelerate==0.34.2 safetensors==0.4.5
!pip install -q fastapi uvicorn[standard] nest-asyncio pyngrok
print("deps installed")


In [ ]:
# Cell 3: load AnimateDiff Lightning 4-step checkpoint.
# Uses ByteDance/AnimateDiff-Lightning with the 4-step distilled UNet -
# cutting diffusion steps to 4 makes 1s-2s clips in ~1-2s per frame on T4.
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler

adapter = MotionAdapter.from_pretrained(
    "ByteDance/AnimateDiff-Lightning",
    cache_dir="/content/hf_cache",
    subfolder="",
    variant="fp16",
    torch_dtype=torch.float16,
)
pipe = AnimateDiffPipeline.from_pretrained(
    "emilianJR/epiCRealism",
    motion_adapter=adapter,
    cache_dir="/content/hf_cache",
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print("model ready on", torch.cuda.get_device_name(0))


In [ ]:
# Cell 4: pyngrok token. REPLACE 'YOUR_NGROK_TOKEN' below with your own.
# Get one free at https://dashboard.ngrok.com/signup -> "Your Authtoken".
# This is a per-user token; never share it in chat or paste in prompts.
NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # <-- replace

# Sanity check
if NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    raise SystemExit("PASTE YOUR NGROK_AUTHTOKEN ABOVE before running this cell.")


In [ ]:
# Cell 5: HTTP server. Identical to the Kaggle notebook but exposed via
# pyngrok so you don't have to copy ngrok binary into the container.
import io, os, time, uuid, base64
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

app = FastAPI()

class GenerateRequest(BaseModel):
    prompt: str
    style: str = "cinematic"
    duration: int = 2        # seconds
    aspect_ratio: str = "16:9"
    seed: int | None = None

@app.get("/health")
async def health():
    import torch
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0),
            "vram_free_mb": torch.cuda.mem_get_info()[0] // 1024**2}

@app.post("/generate")
async def generate(req: GenerateRequest):
    W, H = {"16:9": (1024, 576), "9:16": (576, 1024), "1:1": (768, 768)}.get(
        req.aspect_ratio, (1024, 576)
    )
    frames = max(8, min(24, req.duration * 8))
    gen = torch.Generator("cuda")
    if req.seed is not None:
        gen.manual_seed(req.seed)

    t0 = time.time()
    out = pipe(
        prompt=f"{req.prompt}, {req.style} style, high quality",
        num_frames=frames,
        height=H,
        width=W,
        num_inference_steps=4,
        guidance_scale=1.0,    # Lightning works best with guidance 1.0
        generator=gen,
    )
    secs = round(time.time() - t0, 2)

    vid = out.frames[0]
    bio = io.BytesIO()
    from diffusers.utils import export_to_video
    export_to_video(vid, "out.mp4", fps=8)
    with open("out.mp4", "rb") as f:
        data = f.read()
    os.remove("out.mp4")
    b64 = base64.b64encode(data).decode()
    return JSONResponse({
        "videoBase64": b64,
        "durationSec": req.duration,
        "elapsedSec": secs,
        "format": "mp4",
        "sizeBytes": len(data),
    })

# Spin up ngrok tunnel
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000, "http").public_url
print(f"\nPUBLIC URL: {public_url}")
print("Set this in .env.local: COLAB_VIDEO_URL=" + public_url)
print("Endpoint: POST " + public_url + "/generate  body={prompt, duration, aspect_ratio}")

# Run server in foreground (blocks the cell - this is intentional)
uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


## What you should see

- `ngrok` should print a URL like `https://1a2b-3c4d.ngrok-free.app`
- Cell stays running ("forever"). Do not stop the runtime or you lose the URL.
- Visit `URL/health` in browser — returns GPU + VRAM JSON.

## Test it

From your host shell:

```bash
curl -X POST https://1a2b-3c4d.ngrok-free.app/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt":"a dhaka sunset","duration":2,"aspect_ratio":"16:9"}'
```

First request takes ~30-60s (cold load). Subsequent requests ~10-20s.


## Hot-reload: keep Colab awake

If your session disconnects (it will, eventually), just re-run cells 3-6. The model re-loads in ~30s.

To minimize disconnects:
- Keep Colab tab in foreground (background tabs get killed in 24h)
- Use Chrome, not Firefox
- Disable hardware acceleration offloading in Chrome (Colab flag)
